In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import timezone

# Load CSVs into DataFrames (reuse variable names if re-running)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

tz = pytz.timezone('Asia/Shanghai')

def ts_summary(series):
    # series: numeric epoch seconds
    if series.dropna().empty:
        return {"min_epoch": None, "min_local": None, "max_epoch": None, "max_local": None}
    mn = int(series.min())
    mx = int(series.max())
    mn_dt = pd.to_datetime(mn, unit='s').tz_localize('UTC').tz_convert(tz)
    mx_dt = pd.to_datetime(mx, unit='s').tz_localize('UTC').tz_convert(tz)
    return {
        "min_epoch": mn,
        "min_local": mn_dt.isoformat(),
        "max_epoch": mx,
        "max_local": mx_dt.isoformat()
    }

def top_cmdb_ids(df, limit=20):
    if 'cmdb_id' not in df.columns:
        return []
    uniq = df['cmdb_id'].dropna().unique()
    if len(uniq) == 0:
        return []
    if len(uniq) > limit:
        vc = df['cmdb_id'].value_counts().head(limit)
        # return list of dicts {cmdb_id, count}
        return [{"cmdb_id": idx, "count": int(cnt)} for idx, cnt in vc.items()]
    else:
        return sorted([str(x) for x in uniq])[:limit]

def top_field_names(df, field_col, limit=50):
    if field_col not in df.columns:
        return []
    names = df[field_col].dropna().unique()
    if len(names) == 0:
        return []
    if len(names) > limit:
        # use frequency to pick top names
        vc = df[field_col].value_counts().head(limit)
        return vc.index.tolist()
    else:
        return sorted(names)[:limit]

# Summaries
metric_summary = {
    "total_rows": int(metric_df.shape[0]),
    "timestamp_range": ts_summary(metric_df['timestamp']) if 'timestamp' in metric_df.columns else {},
    "top_20_cmdb_id": top_cmdb_ids(metric_df, limit=20),
    "top_kpi_names": top_field_names(metric_df, 'kpi_name', limit=50)
}

trace_summary = {
    "total_rows": int(trace_df.shape[0]),
    "timestamp_range": ts_summary(trace_df['timestamp']) if 'timestamp' in trace_df.columns else {},
    "top_20_cmdb_id": top_cmdb_ids(trace_df, limit=20),
    "top_trace_names": top_field_names(trace_df, 'trace_name', limit=50)
}

log_summary = {
    "total_rows": int(log_df.shape[0]),
    "timestamp_range": ts_summary(log_df['timestamp']) if 'timestamp' in log_df.columns else {},
    "top_20_cmdb_id": top_cmdb_ids(log_df, limit=20),
    "top_log_names": top_field_names(log_df, 'log_name', limit=50)
}

# For error_logs: top 20 unique message samples shortened to first 200 chars
def top_error_messages(df, limit=20, max_len=200):
    if 'message' not in df.columns:
        return []
    msgs = df['message'].dropna().astype(str)
    if msgs.empty:
        return []
    unique_msgs = msgs.drop_duplicates()
    if unique_msgs.shape[0] > limit:
        # choose top by frequency
        vc = msgs.value_counts().head(limit)
        selected = vc.index.tolist()
    else:
        selected = unique_msgs.tolist()[:limit]
    # shorten
    return [m.replace("\n", " ")[:max_len] for m in selected]

error_summary = {
    "total_rows": int(error_df.shape[0]),
    "timestamp_range": ts_summary(error_df['timestamp']) if 'timestamp' in error_df.columns else {},
    "top_20_cmdb_id": top_cmdb_ids(error_df, limit=20),
    "top_20_message_samples": top_error_messages(error_df, limit=20, max_len=200)
}

result = {
    "metric": metric_summary,
    "trace": trace_summary,
    "log": log_summary,
    "error_logs": error_summary
}

# Display compact result variable
result

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry:\n"
    "- metric.csv: 322,448 rows covering 2022-03-21 10:00 → 11:00 (UTC+8). "
    "Top components by row count include adservice, adservice2, frontend2-0 and frontend-0/1/2. "
    "Top KPI names are dominated by container metrics (many container.node-6.* entries: CPU, memory, fs, network counters).\n"
    "- trace.csv: 35,600 rows covering 2022-03-21 10:30 → 10:59 (UTC+8). "
    "Top components: frontend-0/1/2 and checkoutservice-2/0/1 among others. "
    "Top trace fields are per-source/destination latency/error/row_count (e.g., trace.from_frontend-*.duration_mean/p95/error_rate, trace.self.*).\n"
    "- log.csv: 1,838 rows covering 2022-03-21 10:30 → 11:00 (UTC+8). "
    "Top components each have ~60+ rows (examples: adservice-1, cartservice-0/1/2, currencyservice-0/1/2). "
    "Top log fields: log.error_count and log.row_count.\n"
    "- error_logs.csv: 0 rows (no error log messages available).\n"
    "Overall: large metric coverage (10:00–11:00) with container/node-level KPIs (node-6 prominent), traces and logs focused on a narrower window (10:30–11:00)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'total_rows': 322448, 'timestamp_range': {'min_epoch': 1647828000, 'min_local': '2022-03-21T10:00:00+08:00', 'max_epoch': 1647831600, 'max_local': '2022-03-21T11:00:00+08:00'}, 'top_20_cmdb_id': [{'cmdb_id': 'adservice', 'count': 20801}, {'cmdb_id': 'adservice2', 'count': 20313}, {'cmdb_id': 'frontend2-0', 'count': 12200}, {'cmdb_id': 'frontend-2', 'count': 11468}, {'cmdb_id': 'frontend-1', 'count': 11468}, {'cmdb_id': 'frontend-0', 'count': 11468}, {'cmdb_id': 'checkoutservice2-0', 'count': 9028}, {'cmdb_id': 'checkoutservice-1', 'count': 8784}, {'cmdb_id': 'checkoutservice-2', 'count': 8540}, {'cmdb_id': 'checkoutservice-0', 'count': 8540}, {'cmdb_id': 'productcatalogservice-1', 'count': 7320}, {'cmdb_id': 'productcatalogservice-0', 'count': 7320}, {'cmdb_id': 'productcatalogservice-2', 'count': 7076}, {'cmdb_id': 'productcatalogservice2-0', 'count': 6344}, {'cmdb_id': 'adservice2-0', 'count': 6222}, {'cmdb_id': 'recommendationservice2-0', 'count': 6222}, {'cmdb_id': 'cartservice2-0', 'count': 6100}, {'cmdb_id': 'cartservice-2', 'count': 6100}, {'cmdb_id': 'cartservice-0', 'count': 6100}, {'cmdb_id': 'recommendationservice-2', 'count': 5978}], 'top_kpi_names': ['container.node-6.container_network_transmit_errors.eth0', 'container.node-6.container_network_transmit_packets.eth0', 'container.node-6.container_network_transmit_packets_dropped.eth0', 'container.node-6.container_network_receive_MB.eth0', 'container.node-6.container_network_receive_errors.eth0', 'container.node-6.container_network_receive_packets.eth0', 'container.node-6.container_network_receive_packets_dropped.eth0', 'container.node-6.container_network_transmit_MB.eth0', 'container.node-6.container_spec_cpu_shares', 'container.node-6.container_spec_memory_reservation_limit_MB', 'container.node-6.container_spec_cpu_quota', 'container.node-6.container_spec_memory_swap_limit_MB', 'container.node-6.container_fs_writes./dev/vda1', 'container.node-6.container_fs_writes./dev/vda', 'container.node-6.container_memory_usage_MB', 'container.node-6.container_memory_swap', 'container.node-6.container_fs_limit_MB./dev/vda1', 'container.node-6.container_fs_io_time_weighted_seconds./dev/vda1', 'container.node-6.container_fs_io_time_seconds./dev/vda1', 'container.node-6.container_fs_io_current./dev/vda1', 'container.node-6.container_spec_memory_limit_MB', 'container.node-6.container_fs_inodes_free./dev/vda1', 'container.node-6.container_cpu_cfs_throttled_seconds', 'container.node-6.container_cpu_cfs_throttled_periods', 'container.node-6.container_fs_inodes./dev/vda1', 'container.node-6.container_file_descriptors', 'container.node-6.container_cpu_cfs_periods', 'container.node-6.container_fs_read_seconds./dev/vda1', 'container.node-6.container_start_time_seconds', 'container.node-6.container_spec_cpu_period', 'container.node-6.container_fs_usage_MB./dev/vda1', 'container.node-6.container_fs_write_seconds./dev/vda1', 'container.node-6.container_ulimits_soft.max_open_files', 'container.node-6.container_sockets', 'container.node-6.container_memory_failures.container.pgmajfault', 'container.node-6.container_memory_failures.hierarchy.pgfault', 'container.node-6.container_memory_failures.hierarchy.pgmajfault', 'container.node-6.container_memory_mapped_file', 'container.node-6.container_memory_max_usage_MB', 'container.node-6.container_memory_rss', 'container.node-6.container_threads_max', 'container.node-6.container_threads', 'container.node-6.container_tasks_state.uninterruptible', 'container.node-6.container_tasks_state.stopped', 'container.node-6.container_fs_sector_writes./dev/vda1', 'container.node-6.container_fs_sector_reads./dev/vda1', 'container.node-6.container_tasks_state.sleeping', 'container.node-6.container_memory_working_set_MB', 'container.node-6.container_fs_reads_merged./dev/vda1', 'container.node-6.container_fs_reads_MB./dev/vda']}, 'trace': {'total_rows': 35600, 'timestamp_range': {'min_epoch': 1647829800, 'min_local': '2022-03-21T10:30:00+08:00', 'max_epoch': 1647831540, 'max_local': '2022-03-21T10:59:00+08:00'}, 'top_20_cmdb_id': [{'cmdb_id': 'frontend-0', 'count': 2740}, {'cmdb_id': 'frontend-1', 'count': 2708}, {'cmdb_id': 'frontend-2', 'count': 2680}, {'cmdb_id': 'checkoutservice-2', 'count': 2604}, {'cmdb_id': 'checkoutservice-0', 'count': 2576}, {'cmdb_id': 'checkoutservice-1', 'count': 2480}, {'cmdb_id': 'productcatalogservice-0', 'count': 1188}, {'cmdb_id': 'productcatalogservice-2', 'count': 1184}, {'cmdb_id': 'productcatalogservice-1', 'count': 1168}, {'cmdb_id': 'frontend2-0', 'count': 1072}, {'cmdb_id': 'checkoutservice2-0', 'count': 1008}, {'cmdb_id': 'recommendationservice-0', 'count': 840}, {'cmdb_id': 'recommendationservice-2', 'count': 840}, {'cmdb_id': 'recommendationservice-1', 'count': 840}, {'cmdb_id': 'cartservice-0', 'count': 824}, {'cmdb_id': 'cartservice-2', 'count': 824}, {'cmdb_id': 'cartservice-1', 'count': 816}, {'cmdb_id': 'currencyservice-2', 'count': 712}, {'cmdb_id': 'currencyservice-1', 'count': 712}, {'cmdb_id': 'shippingservice-0', 'count': 708}], 'top_trace_names': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_checkoutservice-2.row_count', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.to_productcatalogservice-0.duration_mean', 'trace.to_productcatalogservice-0.duration_p95', 'trace.to_productcatalogservice-0.error_rate', 'trace.to_productcatalogservice-0.row_count', 'trace.to_productcatalogservice-2.row_count', 'trace.to_productcatalogservice-2.error_rate', 'trace.to_productcatalogservice-2.duration_p95', 'trace.to_productcatalogservice-2.duration_mean', 'trace.to_productcatalogservice-1.row_count', 'trace.to_productcatalogservice-1.error_rate', 'trace.to_productcatalogservice-1.duration_p95', 'trace.to_productcatalogservice-1.duration_mean', 'trace.from_root.row_count', 'trace.from_root.error_rate', 'trace.from_root.duration_p95', 'trace.from_root.duration_mean', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.to_currencyservice-1.row_count', 'trace.to_currencyservice-1.error_rate']}, 'log': {'total_rows': 1838, 'timestamp_range': {'min_epoch': 1647829800, 'min_local': '2022-03-21T10:30:00+08:00', 'max_epoch': 1647831600, 'max_local': '2022-03-21T11:00:00+08:00'}, 'top_20_cmdb_id': [{'cmdb_id': 'adservice-1', 'count': 62}, {'cmdb_id': 'cartservice-1', 'count': 62}, {'cmdb_id': 'cartservice-0', 'count': 62}, {'cmdb_id': 'cartservice-2', 'count': 62}, {'cmdb_id': 'currencyservice-2', 'count': 62}, {'cmdb_id': 'currencyservice-0', 'count': 62}, {'cmdb_id': 'currencyservice-1', 'count': 62}, {'cmdb_id': 'productcatalogservice-2', 'count': 62}, {'cmdb_id': 'recommendationservice-0', 'count': 62}, {'cmdb_id': 'recommendationservice-1', 'count': 62}, {'cmdb_id': 'recommendationservice-2', 'count': 62}, {'cmdb_id': 'shippingservice-0', 'count': 62}, {'cmdb_id': 'shippingservice-1', 'count': 62}, {'cmdb_id': 'frontend-1', 'count': 62}, {'cmdb_id': 'frontend-0', 'count': 62}, {'cmdb_id': 'paymentservice-0', 'count': 60}, {'cmdb_id': 'emailservice-2', 'count': 60}, {'cmdb_id': 'emailservice-0', 'count': 60}, {'cmdb_id': 'emailservice-1', 'count': 60}, {'cmdb_id': 'checkoutservice-2', 'count': 60}], 'top_log_names': ['log.error_count', 'log.row_count']}, 'error_logs': {'total_rows': 0, 'timestamp_range': {'min_epoch': None, 'min_local': None, 'max_epoch': None, 'max_local': None}, 'top_20_cmdb_id': [], 'top_20_message_samples': []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse metric_df loaded previously in the kernel
# Ensure required columns exist
assert {'timestamp','cmdb_id','kpi_name','value'}.issubset(set(metric_df.columns)), "metric_df missing required columns"

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for each (cmdb_id, kpi_name) using the FULL series
thresholds = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# 2) Filter metric.csv to the incident window inclusive: [1647829800, 1647831600]
start_ts = 1647829800
end_ts = 1647831600
df_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into windowed data and detect anomalies where value >= p95 OR value <= p5
df_window = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# If any thresholds are missing (shouldn't happen), drop those rows to avoid false positives
df_window = df_window.dropna(subset=['p95','p5'])

anomalies_df = df_window[(df_window['value'] >= df_window['p95']) | (df_window['value'] <= df_window['p5'])].copy()

# Convert earliest timestamps to local ISO strings helper
def epoch_to_local_iso(series):
    return pd.to_datetime(series, unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# 4A) Per (cmdb_id, kpi_name) anomaly summary in window
if not anomalies_df.empty:
    per_kpi = (
        anomalies_df
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            global_p95=('p95','first'),
            global_p5=('p5','first'),
            anomaly_count_in_window=('value','size'),
            earliest_anom_ts_epoch=('timestamp','min'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
        .sort_values(['anomaly_count_in_window','earliest_anom_ts_epoch'], ascending=[False, True])
    )
    per_kpi['earliest_anom_ts_local'] = epoch_to_local_iso(per_kpi['earliest_anom_ts_epoch'])
    per_kpi_top20 = per_kpi.head(20).copy()
else:
    per_kpi_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','global_p5','anomaly_count_in_window',
        'earliest_anom_ts_epoch','earliest_anom_ts_local','max_value_in_window','min_value_in_window'
    ])

# 4B) Per-component aggregation (sum across KPIs)
if not anomalies_df.empty:
    per_component = (
        anomalies_df
        .groupby('cmdb_id', as_index=False)
        .agg(
            total_anomaly_count=('value','size'),
            number_of_KPIs_with_anomalies=('kpi_name', pd.Series.nunique),
            earliest_anom_ts_epoch=('timestamp','min')
        )
        .sort_values(['total_anomaly_count','earliest_anom_ts_epoch'], ascending=[False, True])
    )
    per_component['earliest_anom_ts_local'] = epoch_to_local_iso(per_component['earliest_anom_ts_epoch'])
    per_component_top20 = per_component.head(20).copy()
else:
    per_component_top20 = pd.DataFrame(columns=[
        'cmdb_id','total_anomaly_count','number_of_KPIs_with_anomalies',
        'earliest_anom_ts_epoch','earliest_anom_ts_local'
    ])

# Prepare JSON-serializable compact outputs (limit to top 20 as requested)
per_kpi_out = per_kpi_top20[
    ['cmdb_id','kpi_name','global_p95','global_p5','anomaly_count_in_window',
     'earliest_anom_ts_epoch','earliest_anom_ts_local','max_value_in_window','min_value_in_window']
].to_dict(orient='records')

per_component_out = per_component_top20[
    ['cmdb_id','total_anomaly_count','number_of_KPIs_with_anomalies',
     'earliest_anom_ts_epoch','earliest_anom_ts_local']
].to_dict(orient='records')

result = {
    "per_kpi_anomalies": per_kpi_out,
    "per_component_anomalies": per_component_out
}

# Display compact result variable
result

```
Out[2]:
```
summary = (
    "Results (metric.csv) — anomaly detection in window 2022-03-21 10:30 → 11:00 (UTC+8):\n"
    "- Method: computed global P95/P5 per (cmdb_id,kpi_name) from the full series, then flagged window values >=P95 or <=P5.\n"
    "- Per-KPI top findings: the top 20 anomalous KPI rows are dominated by adservice (many KPIs such as app.grpc.*, app.http.*, and numerous JVM/runtime metrics). "
    "Many of these KPIs show anomaly_count = 31 (i.e., anomalous at every sampled minute in the 31-minute window) and their values equal the computed P95/P5.\n"
    "- Per-component summary: adservice is the most affected with 9,576 anomalies across 341 KPIs, followed by adservice2 (8,667 anomalies, 333 KPIs). "
    "Frontend pods (frontend2-0, frontend-2, frontend-1, frontend-0) and several checkout/productcatalog/recommendation components also show thousands of anomalies. \n"
    "- Timing: the earliest detected anomalies for all top entries are at 2022-03-21T10:30:00+08:00 (window start).\n"
    "- Practical takeaway: adservice and adservice2 exhibit the largest, broadest anomaly footprint during the incident window (many KPIs anomalous for the entire window). "
    "Recommend focusing investigation on these services (logs, traces, JVM/process state, and upstream/downstream interactions) to determine why many metrics are pinned at threshold values."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'per_kpi_anomalies': [{'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.rr', 'global_p95': 100.0, 'global_p5': 100.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 100.0, 'min_value_in_window': 100.0}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'global_p95': 100.0, 'global_p5': 100.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 100.0, 'min_value_in_window': 100.0}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.count', 'global_p95': 24.0, 'global_p5': 24.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 24.0, 'min_value_in_window': 24.0}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.rr', 'global_p95': 100.0, 'global_p5': 100.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 100.0, 'min_value_in_window': 100.0}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.sr', 'global_p95': 100.0, 'global_p5': 100.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 100.0, 'min_value_in_window': 100.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_LoadedClassCount', 'global_p95': 5162.0, 'global_p5': 5162.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 5162.0, 'min_value_in_window': 5162.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'global_p95': 5185.0, 'global_p5': 5185.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 5185.0, 'min_value_in_window': 5185.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'global_p95': 23.0, 'global_p5': 23.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 23.0, 'min_value_in_window': 23.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_ClassLoading_Verbose', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'global_p95': 1.0, 'global_p5': 1.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 1.0, 'min_value_in_window': 1.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_Compilation_TotalCompilationTime', 'global_p95': 47021.0, 'global_p5': 47021.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 47021.0, 'min_value_in_window': 47021.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'global_p95': 8.0, 'global_p5': 8.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 8.0, 'min_value_in_window': 8.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'global_p95': 480.0, 'global_p5': 480.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 480.0, 'min_value_in_window': 480.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'global_p95': 1.0, 'global_p5': 1.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 1.0, 'min_value_in_window': 1.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'global_p95': 1.0, 'global_p5': 1.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 1.0, 'min_value_in_window': 1.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'global_p95': 94.0, 'global_p5': 94.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 94.0, 'min_value_in_window': 94.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'global_p95': 539245384.0, 'global_p5': 539245384.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 539245384.0, 'min_value_in_window': 539245384.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'global_p95': 8.0, 'global_p5': 8.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 8.0, 'min_value_in_window': 8.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'global_p95': 23330816.0, 'global_p5': 23330816.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 23330816.0, 'min_value_in_window': 23330816.0}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'global_p95': 23330816.0, 'global_p5': 23330816.0, 'anomaly_count_in_window': 31, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 23330816.0, 'min_value_in_window': 23330816.0}], 'per_component_anomalies': [{'cmdb_id': 'adservice', 'total_anomaly_count': 9576, 'number_of_KPIs_with_anomalies': 341, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'adservice2', 'total_anomaly_count': 8667, 'number_of_KPIs_with_anomalies': 333, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend2-0', 'total_anomaly_count': 4404, 'number_of_KPIs_with_anomalies': 200, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend-2', 'total_anomaly_count': 4091, 'number_of_KPIs_with_anomalies': 188, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend-1', 'total_anomaly_count': 4057, 'number_of_KPIs_with_anomalies': 187, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend-0', 'total_anomaly_count': 4022, 'number_of_KPIs_with_anomalies': 187, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'checkoutservice2-0', 'total_anomaly_count': 3185, 'number_of_KPIs_with_anomalies': 148, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'checkoutservice-1', 'total_anomaly_count': 2950, 'number_of_KPIs_with_anomalies': 144, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'checkoutservice-2', 'total_anomaly_count': 2874, 'number_of_KPIs_with_anomalies': 140, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'checkoutservice-0', 'total_anomaly_count': 2816, 'number_of_KPIs_with_anomalies': 140, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'productcatalogservice-1', 'total_anomaly_count': 2634, 'number_of_KPIs_with_anomalies': 120, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'productcatalogservice-0', 'total_anomaly_count': 2594, 'number_of_KPIs_with_anomalies': 120, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'productcatalogservice-2', 'total_anomaly_count': 2525, 'number_of_KPIs_with_anomalies': 116, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'recommendationservice2-0', 'total_anomaly_count': 2514, 'number_of_KPIs_with_anomalies': 102, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'adservice2-0', 'total_anomaly_count': 2455, 'number_of_KPIs_with_anomalies': 102, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'recommendationservice-1', 'total_anomaly_count': 2393, 'number_of_KPIs_with_anomalies': 98, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'recommendationservice-2', 'total_anomaly_count': 2393, 'number_of_KPIs_with_anomalies': 98, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'recommendationservice-0', 'total_anomaly_count': 2384, 'number_of_KPIs_with_anomalies': 98, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'productcatalogservice2-0', 'total_anomaly_count': 2356, 'number_of_KPIs_with_anomalies': 104, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'cartservice-2', 'total_anomaly_count': 2307, 'number_of_KPIs_with_anomalies': 100, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}]}```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse trace_df from kernel
assert {'timestamp','cmdb_id','trace_name','value'}.issubset(set(trace_df.columns)), "trace_df missing required columns"

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for each (cmdb_id, trace_name) using the full series (must be done BEFORE filtering)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p5=lambda x: x.quantile(0.05))
)

# 2) Filter trace.csv to the incident window inclusive
start_ts = 1647829800
end_ts = 1647831600
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Restrict to trace_name values containing substrings: 'error_rate', 'duration', or 'row_count'
substrs = ['error_rate','duration','row_count']
mask = trace_window['trace_name'].str.lower().str.contains('|'.join(substrs))
trace_window_filtered = trace_window[mask].copy()

# 4) Merge thresholds and flag anomalies where value >= global_p95 OR value <= global_p5
trace_window_filtered = trace_window_filtered.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
trace_window_filtered = trace_window_filtered.dropna(subset=['global_p95','global_p5'])
anom_mask = (trace_window_filtered['value'] >= trace_window_filtered['global_p95']) | (trace_window_filtered['value'] <= trace_window_filtered['global_p5'])
trace_anomalies = trace_window_filtered[anom_mask].copy()

# helper to convert epoch to local ISO
def epoch_to_local_iso(series):
    return pd.to_datetime(series, unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# 5A) Top 20 (cmdb_id, trace_name) by anomaly_count_in_window
if not trace_anomalies.empty:
    per_trace = (
        trace_anomalies
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            global_p95=('global_p95','first'),
            global_p5=('global_p5','first'),
            anomaly_count_in_window=('value','size'),
            earliest_anom_ts_epoch=('timestamp','min'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
        .sort_values(['anomaly_count_in_window','earliest_anom_ts_epoch'], ascending=[False, True])
    )
    per_trace['earliest_anom_ts_local'] = epoch_to_local_iso(per_trace['earliest_anom_ts_epoch'])
    per_trace_top20 = per_trace.head(20).copy()
else:
    per_trace_top20 = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_p95','global_p5','anomaly_count_in_window',
        'earliest_anom_ts_epoch','earliest_anom_ts_local','max_value_in_window','min_value_in_window'
    ])

# 5B) Top 20 cmdb_id aggregated by total trace anomaly_count across trace_names
if not trace_anomalies.empty:
    per_component = (
        trace_anomalies
        .groupby('cmdb_id', as_index=False)
        .agg(
            total_trace_anomaly_count=('value','size'),
            number_of_trace_names_with_anomalies=('trace_name', pd.Series.nunique),
            earliest_anom_ts_epoch=('timestamp','min')
        )
        .sort_values(['total_trace_anomaly_count','earliest_anom_ts_epoch'], ascending=[False, True])
    )
    per_component['earliest_anom_ts_local'] = epoch_to_local_iso(per_component['earliest_anom_ts_epoch'])
    per_component_top20 = per_component.head(20).copy()
else:
    per_component_top20 = pd.DataFrame(columns=[
        'cmdb_id','total_trace_anomaly_count','number_of_trace_names_with_anomalies',
        'earliest_anom_ts_epoch','earliest_anom_ts_local'
    ])

# Prepare compact JSON-like outputs (limit to top 20)
per_trace_out = per_trace_top20[
    ['cmdb_id','trace_name','global_p95','global_p5','anomaly_count_in_window',
     'earliest_anom_ts_epoch','earliest_anom_ts_local','max_value_in_window','min_value_in_window']
].to_dict(orient='records')

per_component_out = per_component_top20[
    ['cmdb_id','total_trace_anomaly_count','number_of_trace_names_with_anomalies',
     'earliest_anom_ts_epoch','earliest_anom_ts_local']
].to_dict(orient='records')

result = {
    "per_trace_anomalies": per_trace_out,
    "per_component_anomalies": per_component_out
}

# Display compact result variable
result

```
Out[3]:
```
Summary of trace-based anomaly detection (window 2022-03-21 10:30 → 11:00 UTC+8):

- Method recap: computed global P95/P5 per (cmdb_id,trace_name) from the full trace series, filtered to the incident window, restricted to trace_name containing "error_rate", "duration", or "row_count", and flagged samples >=P95 or <=P5.

- Key per-trace findings (top examples):
  - Many trace metrics for adservice-* and cartservice-* (e.g., trace.from_frontend-*.error_rate) show anomaly_count = 30 (i.e., anomalous for the whole window). Their global P95 and P5 are 0.0 and window values are 0.0.
  - Several trace.self.duration_p95 / trace.self.error_rate entries for cartservice pods are also flagged for every minute in the window.

- Key per-component findings:
  - Highest aggregated trace anomaly counts: checkoutservice pods (checkoutservice-0/2/1) and frontend pods (frontend-0/2/1) are the most affected (each with >1k trace-anomaly events in the window).
  - Other affected components with hundreds of anomalies: cartservice-*, productcatalogservice-*, checkoutservice2-0, frontend2-0, recommendationservice-*, shippingservice-*.
  - For all top entries the earliest anomaly time is 2022-03-21T10:30:00+08:00 (window start).

- Interpretation / likely explanations:
  - Many error_rate/duration/row_count traces are pinned at 0.0 (global P95/P5 = 0.0), so the detector flagged values equal to that threshold throughout the window. This pattern often indicates either:
    - real drop in traffic / no errors (zero error_rate or zero requests) across many services, or
    - telemetry/tracing collection interruption or sampling change (instrumentation not reporting), producing zeroed metrics.
  - The broad footprint across frontend, checkout, and checkout-adjacent services suggests a systemic issue impacting request flow or tracing capture rather than a localized CPU/memory problem.

- Recommended next steps:
  1. Check request/traffic rates (frontend ingress, load balancer, API gateway) to confirm if traffic dropped at 10:30.
  2. Inspect tracing/telemetry pipeline and agent logs (collector/agent) for failures or config changes around 10:30 that could cause many zeroed traces.
  3. Review application logs (frontend, checkout, adservice) and service error logs for errors or connection failures at the same time.
  4. If traffic is present but traces are zero, focus on tracing agent / network / sampling settings; if traffic dropped, investigate upstream (load balancer, CDN, synthetic test).
  5. Prioritize checkout and frontend pods and adservice/adservice2 for investigation (highest anomaly volumes).

- Bottom line: trace anomalies are widespread and start at the window start (10:30). They likely reflect either a real traffic drop or a telemetry collection problem; investigate traffic and tracing/agent health first.

The original code execution output of IPython Kernel is also provided below for reference:

{'per_trace_anomalies': [{'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-1.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-2.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-1', 'trace_name': 'trace.from_frontend-0.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-1', 'trace_name': 'trace.from_frontend-1.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-1', 'trace_name': 'trace.from_frontend-2.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-0.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-1.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'adservice2-0', 'trace_name': 'trace.from_frontend2-0.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_frontend-0.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_frontend-1.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_frontend-2.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-0', 'trace_name': 'trace.self.duration_p95', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-0', 'trace_name': 'trace.self.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-1', 'trace_name': 'trace.from_frontend-0.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-1', 'trace_name': 'trace.from_frontend-1.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-1', 'trace_name': 'trace.from_frontend-2.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-1', 'trace_name': 'trace.self.duration_p95', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}, {'cmdb_id': 'cartservice-1', 'trace_name': 'trace.self.error_rate', 'global_p95': 0.0, 'global_p5': 0.0, 'anomaly_count_in_window': 30, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800', 'max_value_in_window': 0.0, 'min_value_in_window': 0.0}], 'per_component_anomalies': [{'cmdb_id': 'checkoutservice-0', 'total_trace_anomaly_count': 1356, 'number_of_trace_names_with_anomalies': 92, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'checkoutservice-2', 'total_trace_anomaly_count': 1330, 'number_of_trace_names_with_anomalies': 92, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'checkoutservice-1', 'total_trace_anomaly_count': 1266, 'number_of_trace_names_with_anomalies': 92, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend-0', 'total_trace_anomaly_count': 1156, 'number_of_trace_names_with_anomalies': 92, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend-2', 'total_trace_anomaly_count': 1141, 'number_of_trace_names_with_anomalies': 92, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend-1', 'total_trace_anomaly_count': 1104, 'number_of_trace_names_with_anomalies': 92, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'cartservice-1', 'total_trace_anomaly_count': 565, 'number_of_trace_names_with_anomalies': 28, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'cartservice-2', 'total_trace_anomaly_count': 564, 'number_of_trace_names_with_anomalies': 28, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'cartservice-0', 'total_trace_anomaly_count': 556, 'number_of_trace_names_with_anomalies': 28, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'productcatalogservice-1', 'total_trace_anomaly_count': 494, 'number_of_trace_names_with_anomalies': 40, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'productcatalogservice-0', 'total_trace_anomaly_count': 484, 'number_of_trace_names_with_anomalies': 40, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'checkoutservice2-0', 'total_trace_anomaly_count': 472, 'number_of_trace_names_with_anomalies': 36, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'productcatalogservice-2', 'total_trace_anomaly_count': 469, 'number_of_trace_names_with_anomalies': 40, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'frontend2-0', 'total_trace_anomaly_count': 419, 'number_of_trace_names_with_anomalies': 36, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'recommendationservice-0', 'total_trace_anomaly_count': 342, 'number_of_trace_names_with_anomalies': 28, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'recommendationservice-2', 'total_trace_anomaly_count': 342, 'number_of_trace_names_with_anomalies': 28, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'recommendationservice-1', 'total_trace_anomaly_count': 332, 'number_of_trace_names_with_anomalies': 28, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'shippingservice-0', 'total_trace_anomaly_count': 283, 'number_of_trace_names_with_anomalies': 24, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'shippingservice-1', 'total_trace_anomaly_count': 282, 'number_of_trace_names_with_anomalies': 24, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}, {'cmdb_id': 'shippingservice-2', 'total_trace_anomaly_count': 271, 'number_of_trace_names_with_anomalies': 24, 'earliest_anom_ts_epoch': 1647829800, 'earliest_anom_ts_local': '2022-03-21T10:30:00+0800'}]}```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Identify trace_name values that contain 'row_count' (from full trace_df)
row_count_names = sorted(trace_df['trace_name'].dropna().unique())
row_count_names = [n for n in row_count_names if 'row_count' in n]

# 2) Compute global P5 for each (cmdb_id, trace_name) using the full series (do NOT filter first)
thresholds_p5 = (
    trace_df[trace_df['trace_name'].isin(row_count_names)]
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .quantile(0.05)
    .rename(columns={'value':'global_p5'})
)

# 3) Filter trace.csv to the incident window inclusive
start_ts = 1647829800
end_ts = 1647831600
trace_window = trace_df[
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts) &
    (trace_df['trace_name'].isin(row_count_names))
].copy()

# 4) Merge global_p5 and mark points where value <= global_p5 OR value == 0.0
trace_window = trace_window.merge(thresholds_p5, on=['cmdb_id','trace_name'], how='left')
trace_window = trace_window.dropna(subset=['global_p5'])  # drop if no global p5 computed
trace_window['marked'] = (trace_window['value'] <= trace_window['global_p5']) | (trace_window['value'] == 0.0)

# Keep only marked rows for interval compression
marked_df = trace_window[trace_window['marked']].copy()

# 5) Compress consecutive marked points into contiguous intervals per (cmdb_id, trace_name)
if not marked_df.empty:
    # ensure sorting
    marked_df = marked_df.sort_values(['cmdb_id','trace_name','timestamp']).reset_index(drop=True)
    # For each group compute difference to previous timestamp
    marked_df['prev_ts'] = marked_df.groupby(['cmdb_id','trace_name'])['timestamp'].shift(1)
    marked_df['ts_diff'] = marked_df['timestamp'] - marked_df['prev_ts']
    # define new interval when ts_diff != 60 (one minute) or prev_ts is NaN
    marked_df['new_interval'] = ((marked_df['ts_diff'] != 60) | (marked_df['prev_ts'].isna())).astype(int)
    # cumulative sum to assign interval ids within each group
    marked_df['interval_id'] = marked_df.groupby(['cmdb_id','trace_name'])['new_interval'].cumsum()

    # Aggregate intervals
    intervals = (
        marked_df
        .groupby(['cmdb_id','trace_name','interval_id'], as_index=False)
        .agg(
            interval_start_epoch=('timestamp','min'),
            interval_end_epoch=('timestamp','max'),
            duration_minutes=('timestamp','size'),
            min_value_in_interval=('value','min'),
            max_value_in_interval=('value','max')
        )
        .sort_values(['interval_start_epoch','cmdb_id','trace_name'])
        .reset_index(drop=True)
    )
else:
    intervals = pd.DataFrame(columns=[
        'cmdb_id','trace_name','interval_id','interval_start_epoch','interval_end_epoch',
        'duration_minutes','min_value_in_interval','max_value_in_interval'
    ])

# 6A) Top 50 intervals sorted by interval_start_epoch (earliest first)
intervals_top50 = intervals.sort_values('interval_start_epoch', ascending=True).head(50)
intervals_top50 = intervals_top50[[
    'cmdb_id','trace_name','interval_start_epoch','interval_end_epoch',
    'duration_minutes','min_value_in_interval','max_value_in_interval'
]]

# 6B) Per-cmdb_id summary: number_of_intervals, earliest_interval_start_epoch, total_minutes_in_intervals
if not intervals.empty:
    per_component = (
        intervals
        .groupby('cmdb_id', as_index=False)
        .agg(
            number_of_intervals=('duration_minutes','size'),
            earliest_interval_start_epoch=('interval_start_epoch','min'),
            total_minutes_in_intervals=('duration_minutes','sum')
        )
        .sort_values('total_minutes_in_intervals', ascending=False)
    )
    per_component_top20 = per_component.head(20)
else:
    per_component_top20 = pd.DataFrame(columns=[
        'cmdb_id','number_of_intervals','earliest_interval_start_epoch','total_minutes_in_intervals'
    ])

# Display compact results (DataFrames)
intervals_top50, per_component_top20

```
Out[4]:
```
Summary of row_count trace intervals (window 2022-03-21 10:30 → 11:00 UTC+8):

What was done
- Identified all trace_name values containing "row_count" from the full trace series.
- Computed global P5 per (cmdb_id, trace_name) using the full trace.csv.
- In the incident window, marked points where value <= global_P5 or value == 0.0 and compressed consecutive marked minutes into contiguous intervals (faults).

Top-50 intervals (earliest first) — examples and pattern
- Intervals start predominantly at the window start (epoch 1647829800 ≙ 2022-03-21T10:30:00+08:00).
- Example intervals from the top-50 (all start at or near window start):
  - checkoutservice-2, trace.to_emailservice-1.row_count: start 1647829800, end 1647829920, duration 3 minutes, values 1.0–1.0
  - checkoutservice-2, trace.to_paymentservice-0.row_count: start 1647829800, end 1647830100, duration 6 minutes, values 1.0–1.0
  - checkoutservice2-0, trace.self.row_count: start 1647829800, end 1647829800, duration 1 minute, value 8.0
  - paymentservice-0, trace.from_checkoutservice-2.row_count: start 1647829800, end 1647830100, duration 6 minutes, values 1.0–1.0
  - Many other traces across checkout, payment, productcatalog, frontend, cartservice, recommendationservice, shippingservice appear with short intervals (1–6 minutes) where row_count is very low (often 1.0) or zero.

Per-component summary (top 20 by total minutes of intervals)
- checkoutservice-0: total_minutes_in_intervals = 233 (132 intervals), earliest start 1647829800
- checkoutservice-2: total_minutes = 197 (114 intervals)
- checkoutservice-1: total_minutes = 193 (129 intervals)
- frontend-2: total_minutes = 115 (68 intervals)
- frontend-0: total_minutes = 107 (91 intervals)
- frontend-1: total_minutes = 91 (76 intervals)
- checkoutservice2-0: total_minutes = 90 (90 intervals)
- paymentservice-0: total_minutes = 53 (22 intervals)
- emailservice-2: total_minutes = 52 (20 intervals)
- productcatalogservice-1: total_minutes = 50 (40 intervals)
- (and others down to frontend2-0 with 30 minutes)
- Most components’ earliest interval start is 1647829800 (window start). A few have earliest start slightly later (example: cartservice-1 earliest 1647829860).

Interpretation
- Many row_count traces show short but widespread intervals of very low or zero traffic beginning at the window start. Checkout-related services and frontends show the largest aggregate impact by minutes.
- This pattern could indicate: a real drop/interruption in request traffic on many call paths (frontends → checkout → downstream services), or an instrumentation/collection problem that zeroed or reduced row_count reporting for multiple services simultaneously.

Recommended next steps
1. Check request ingress / load balancer / gateway metrics around 10:30 to confirm whether overall traffic dropped.
2. Inspect application logs for frontends and checkout pods for errors or connection failures beginning at 10:30.
3. Check tracing/telemetry agents and collectors for errors or config changes around 10:30 that could cause low/zero row_count reporting.
4. Prioritize investigation on checkoutservice (highest total minutes), then frontend pods and payment/productcatalog services.

Bottom line
- There are many short, concurrent intervals of very low/zero row_count across the system starting at 10:30, with checkoutservice and frontend pods most affected. Investigate traffic flow and tracing/collection health first.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                               trace_name  interval_start_epoch  interval_end_epoch  duration_minutes  min_value_in_interval  max_value_in_interval
19   checkoutservice-2        trace.to_emailservice-1.row_count            1647829800          1647829920                 3                    1.0                    1.0
20   checkoutservice-2      trace.to_paymentservice-0.row_count            1647829800          1647830100                 6                    1.0                    1.0
21  checkoutservice2-0         trace.from_frontend2-0.row_count            1647829800          1647829800                 1                    1.0                    1.0
22  checkoutservice2-0                trace.from_root.row_count            1647829800          1647829800                 1                    3.0                    3.0
23  checkoutservice2-0                     trace.self.row_count            1647829800          1647829800                 1                    8.0                    8.0
..                 ...                                      ...                   ...                 ...               ...                    ...                    ...
57   paymentservice2-0  trace.from_checkoutservice2-0.row_count            1647829800          1647829800                 1                    1.0                    1.0
56    paymentservice-2   trace.from_checkoutservice-0.row_count            1647829800          1647829920                 3                    1.0                    1.0
55    paymentservice-1   trace.from_checkoutservice-1.row_count            1647829800          1647829980                 4                    1.0                    1.0
54    paymentservice-0   trace.from_checkoutservice-2.row_count            1647829800          1647830100                 6                    1.0                    1.0
53    paymentservice-0   trace.from_checkoutservice-1.row_count            1647829800          1647829920                 3                    1.0                    1.0

[50 rows x 7 columns],                     cmdb_id  number_of_intervals  earliest_interval_start_epoch  total_minutes_in_intervals
8         checkoutservice-0                  132                     1647829800                         233
10        checkoutservice-2                  114                     1647829800                         197
9         checkoutservice-1                  129                     1647829800                         193
22               frontend-2                   68                     1647829800                         115
20               frontend-0                   91                     1647829800                         107
21               frontend-1                   76                     1647829800                          91
11       checkoutservice2-0                   90                     1647829800                          90
24         paymentservice-0                   22                     1647829800                          53
18           emailservice-2                   20                     1647829800                          52
29  productcatalogservice-1                   40                     1647829800                          50
25         paymentservice-1                   20                     1647829800                          47
26         paymentservice-2                   20                     1647829800                          45
16           emailservice-0                   19                     1647829800                          44
17           emailservice-1                   20                     1647829800                          44
30  productcatalogservice-2                   39                     1647829800                          41
28  productcatalogservice-0                   34                     1647829800                          38
37        shippingservice-1                   29                     1647829800                          34
5             cartservice-1                   28                     1647829860                          33
33  recommendationservice-1                   27                     1647829800                          31
23              frontend2-0                   30                     1647829800                          30)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Filter to incident window inclusive
start_ts = 1647829800
end_ts = 1647831600

# 2) Select trace records per the condition
mask_time = trace_df['timestamp'].between(start_ts, end_ts)
mask_row_count = trace_df['trace_name'].str.contains('row_count', na=False)
mask_froms = trace_df['trace_name'].str.contains('from_root|from_frontend|from_frontend2', na=False)
mask_frontend_self = trace_df['cmdb_id'].fillna('').str.startswith('frontend') & (trace_df['trace_name'] == 'trace.self.row_count')

mask = mask_time & mask_row_count & (mask_froms | mask_frontend_self)

df_sel = trace_df[mask].copy()

# 3) Aggregate by timestamp (sum values) to compute total_ingress_row_count_per_minute
agg = df_sel.groupby('timestamp', as_index=False)['value'].sum().rename(columns={'value':'total_ingress_row_count'})

# Ensure we have a row for every minute in the window (fill missing minutes with 0)
full_ts = pd.DataFrame({'timestamp': np.arange(start_ts, end_ts + 1, 60)})
df_ingress = full_ts.merge(agg, on='timestamp', how='left').fillna({'total_ingress_row_count': 0})
df_ingress['total_ingress_row_count'] = df_ingress['total_ingress_row_count'].astype(float)

# Add local time string in UTC+8
df_ingress['local_time'] = pd.to_datetime(df_ingress['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# Reorder columns and sort ascending by timestamp
df_ingress = df_ingress[['timestamp','local_time','total_ingress_row_count']].sort_values('timestamp').reset_index(drop=True)

# 4) Small summary: 5 minutes with the lowest total_ingress_row_count
df_min5 = df_ingress.nsmallest(5, ['total_ingress_row_count','timestamp']).reset_index(drop=True)

# Display compact results (per-minute series up to 31 rows, and 5-row min-summary)
df_ingress, df_min5

```
Out[5]:
```
Summary (trace-based ingress row_count aggregated per minute, window 2022-03-21 10:30 → 11:00 UTC+8):

- Per-minute series (31 minutes): total_ingress_row_count ranges roughly from ~6.6k to ~8.0k for most minutes. The series starts 2022-03-21T10:30:00+08:00 and ends 2022-03-21T11:00:00+08:00.

- Notable anomaly: the final minute (1647831600 → 2022-03-21T11:00:00+08:00) has total_ingress_row_count = 0.0, while prior minutes show thousands of rows. This is an abrupt drop at the window end.

- Five minutes with the lowest totals (epoch, local time, total):
  1. 1647831600 — 2022-03-21T11:00:00+08:00 — 0.0
  2. 1647829860 — 2022-03-21T10:31:00+08:00 — 6,652.0
  3. 1647829980 — 2022-03-21T10:33:00+08:00 — 6,665.0
  4. 1647829800 — 2022-03-21T10:30:00+08:00 — 6,747.0
  5. 1647830100 — 2022-03-21T10:35:00+08:00 — 6,769.0

Interpretation / recommended checks:
- The consistent ~6.6k–8.0k per-minute totals indicate normal ingress traffic for most of the window.
- The sudden zero at 11:00 likely indicates either (a) a real traffic drop to zero at that minute, or (b) telemetry/tracing collection interruption (agent/collector or export issue).
- Immediate actions: check ingress/load-balancer metrics and application logs around 11:00, and inspect tracing/telemetry agent and collector status for errors or restarts at that time.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp                local_time  total_ingress_row_count
0   1647829800  2022-03-21T10:30:00+0800                   6747.0
1   1647829860  2022-03-21T10:31:00+0800                   6652.0
2   1647829920  2022-03-21T10:32:00+0800                   6873.0
3   1647829980  2022-03-21T10:33:00+0800                   6665.0
4   1647830040  2022-03-21T10:34:00+0800                   6823.0
..         ...                       ...                      ...
26  1647831360  2022-03-21T10:56:00+0800                   7864.0
27  1647831420  2022-03-21T10:57:00+0800                   8020.0
28  1647831480  2022-03-21T10:58:00+0800                   8031.0
29  1647831540  2022-03-21T10:59:00+0800                   7093.0
30  1647831600  2022-03-21T11:00:00+0800                      0.0

[31 rows x 3 columns],     timestamp                local_time  total_ingress_row_count
0  1647831600  2022-03-21T11:00:00+0800                      0.0
1  1647829860  2022-03-21T10:31:00+0800                   6652.0
2  1647829980  2022-03-21T10:33:00+0800                   6665.0
3  1647829800  2022-03-21T10:30:00+0800                   6747.0
4  1647830100  2022-03-21T10:35:00+0800                   6769.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse log_df from kernel and timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Filter to window inclusive
start_ts = 1647829800
end_ts = 1647831600
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 2) Consider only the two log_name values
target_names = ['log.error_count','log.row_count']
log_window = log_window[log_window['log_name'].isin(target_names)].copy()

# Helper to aggregate for a specific log_name
def aggregate_for_logname(df, logname):
    sub = df[df['log_name'] == logname].copy()
    cols = ['cmdb_id','sum_value','count_nonzero','earliest_nonzero_ts_epoch','earliest_nonzero_ts_local','min_value_in_window','max_value_in_window']
    if sub.empty:
        return pd.DataFrame(columns=cols)
    # sum_value
    agg_sum = sub.groupby('cmdb_id', as_index=False)['value'].sum().rename(columns={'value':'sum_value'})
    # count_nonzero
    agg_countnz = sub.groupby('cmdb_id')['value'].apply(lambda s: int((s>0).sum())).reset_index(name='count_nonzero')
    # earliest_nonzero_timestamp (epoch) — null if none
    nz = sub[sub['value']>0].groupby('cmdb_id', as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_nonzero_ts_epoch'})
    # min/max in window
    agg_min = sub.groupby('cmdb_id', as_index=False)['value'].min().rename(columns={'value':'min_value_in_window'})
    agg_max = sub.groupby('cmdb_id', as_index=False)['value'].max().rename(columns={'value':'max_value_in_window'})
    # merge
    out = agg_sum.merge(agg_countnz, on='cmdb_id', how='left') \
                 .merge(nz, on='cmdb_id', how='left') \
                 .merge(agg_min, on='cmdb_id', how='left') \
                 .merge(agg_max, on='cmdb_id', how='left')
    # convert earliest ts to local iso (keep epoch too). If NaN, set local to None
    out['earliest_nonzero_ts_local'] = pd.to_datetime(out['earliest_nonzero_ts_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    out.loc[out['earliest_nonzero_ts_epoch'].isna(), 'earliest_nonzero_ts_local'] = None
    # reorder & sort by sum_value desc, limit top 20
    out = out[['cmdb_id','sum_value','count_nonzero','earliest_nonzero_ts_epoch','earliest_nonzero_ts_local','min_value_in_window','max_value_in_window']]
    out = out.sort_values('sum_value', ascending=False).head(20).reset_index(drop=True)
    out['sum_value'] = out['sum_value'].astype(float)
    out['count_nonzero'] = out['count_nonzero'].fillna(0).astype(int)
    return out

# 3 & 4) Produce two compact lists (DataFrames) limited to top 20 by sum_value
error_count_summary = aggregate_for_logname(log_window, 'log.error_count')
row_count_summary = aggregate_for_logname(log_window, 'log.row_count')

# Display compact results
error_count_summary, row_count_summary

```
Out[6]:
```
summary = (
    "Log summary for window 2022-03-21 10:30 → 11:00 (UTC+8):\n\n"
    "1) log.error_count:\n"
    "- Top components (by sum) all have sum_value = 0 and count_nonzero = 0. In other words, no error_count entries > 0 were observed for the listed components during the window.\n\n"
    "2) log.row_count:\n"
    "- Frontends show the highest aggregated row counts: frontend-1 (sum=49,192, min=9, max=1,878), frontend-0 (sum=48,392, min=12, max=1,920), frontend-2 (sum=34,393, min=861, max=1,296).\n"
    "- Productcatalogservice instances are also high (~31k sums each). Cartservice and currencyservice pods show strong per-minute traffic (~27k and ~24k total respectively). Recommendationservice and adservice pods have lower totals (several thousand).\n"
    "- For all top components, count_nonzero = 30–31 and earliest_nonzero_timestamp = 1647829800 (2022-03-21T10:30:00+08:00), meaning row_count > 0 was present from the window start and persisted every minute.\n\n"
    "Interpretation and next steps:\n"
    "- Aggregated logs show no recorded errors (log.error_count = 0) while row_count indicates steady request/processing activity across frontends and downstream services throughout the window.\n"
    "- If other telemetry (metrics/traces) suggests problems, they are unlikely to be captured as application-level errors in logs; investigate tracing/metric collection, downstream timeouts, or infrastructure (network/agent) issues.\n"
    "- Recommended checks: inspect detailed application logs for individual error messages (not just error_count aggregations), review trace spans and metric anomalies, and verify telemetry agent/collector health around the window start."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  sum_value  count_nonzero  earliest_nonzero_ts_epoch earliest_nonzero_ts_local  min_value_in_window  max_value_in_window
0         adservice-0        0.0              0                        NaN                      None                  0.0                  0.0
1         adservice-1        0.0              0                        NaN                      None                  0.0                  0.0
2         adservice-2        0.0              0                        NaN                      None                  0.0                  0.0
3       cartservice-0        0.0              0                        NaN                      None                  0.0                  0.0
4       cartservice-1        0.0              0                        NaN                      None                  0.0                  0.0
5       cartservice-2        0.0              0                        NaN                      None                  0.0                  0.0
6   checkoutservice-0        0.0              0                        NaN                      None                  0.0                  0.0
7   checkoutservice-1        0.0              0                        NaN                      None                  0.0                  0.0
8   checkoutservice-2        0.0              0                        NaN                      None                  0.0                  0.0
9   currencyservice-0        0.0              0                        NaN                      None                  0.0                  0.0
10  currencyservice-1        0.0              0                        NaN                      None                  0.0                  0.0
11  currencyservice-2        0.0              0                        NaN                      None                  0.0                  0.0
12     emailservice-0        0.0              0                        NaN                      None                  0.0                  0.0
13     emailservice-1        0.0              0                        NaN                      None                  0.0                  0.0
14     emailservice-2        0.0              0                        NaN                      None                  0.0                  0.0
15         frontend-0        0.0              0                        NaN                      None                  0.0                  0.0
16         frontend-1        0.0              0                        NaN                      None                  0.0                  0.0
17         frontend-2        0.0              0                        NaN                      None                  0.0                  0.0
18   paymentservice-0        0.0              0                        NaN                      None                  0.0                  0.0
19   paymentservice-1        0.0              0                        NaN                      None                  0.0                  0.0,                     cmdb_id  sum_value  count_nonzero  earliest_nonzero_ts_epoch earliest_nonzero_ts_local  min_value_in_window  max_value_in_window
0                frontend-1    49192.0             31                 1647829800  2022-03-21T10:30:00+0800                  9.0               1878.0
1                frontend-0    48392.0             31                 1647829800  2022-03-21T10:30:00+0800                 12.0               1920.0
2                frontend-2    34393.0             30                 1647829800  2022-03-21T10:30:00+0800                861.0               1296.0
3   productcatalogservice-2    31321.0             31                 1647829800  2022-03-21T10:30:00+0800                177.0               1159.0
4   productcatalogservice-0    31106.0             30                 1647829800  2022-03-21T10:30:00+0800                904.0               1159.0
5   productcatalogservice-1    31101.0             30                 1647829800  2022-03-21T10:30:00+0800                920.0               1164.0
6             cartservice-2    27718.0             31                 1647829800  2022-03-21T10:30:00+0800                 18.0               1091.0
7             cartservice-1    27717.0             31                 1647829800  2022-03-21T10:30:00+0800                 27.0               1045.0
8             cartservice-0    27713.0             31                 1647829800  2022-03-21T10:30:00+0800                 18.0               1036.0
9         currencyservice-0    24043.0             31                 1647829800  2022-03-21T10:30:00+0800                  9.0               1086.0
10        currencyservice-1    24011.0             31                 1647829800  2022-03-21T10:30:00+0800                 11.0               1103.0
11        currencyservice-2    23999.0             31                 1647829800  2022-03-21T10:30:00+0800                 10.0               1086.0
12  recommendationservice-1     6610.0             31                 1647829800  2022-03-21T10:30:00+0800                  1.0                256.0
13  recommendationservice-0     6608.0             31                 1647829800  2022-03-21T10:30:00+0800                  2.0                261.0
14  recommendationservice-2     6605.0             31                 1647829800  2022-03-21T10:30:00+0800                  3.0                255.0
15              adservice-0     3846.0             30                 1647829800  2022-03-21T10:30:00+0800                110.0                164.0
16              adservice-1     3500.0             31                 1647829800  2022-03-21T10:30:00+0800                 16.0                156.0
17              adservice-2     3484.0             30                 1647829800  2022-03-21T10:30:00+0800                 92.0                154.0
18        shippingservice-0     2977.0             31                 1647829800  2022-03-21T10:30:00+0800                  4.0                152.0
19        shippingservice-2     2977.0             30                 1647829800  2022-03-21T10:30:00+0800                 46.0                163.0)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Timestamps
ts_1059 = 1647831540  # 2022-03-21T10:59:00+08:00
ts_1100 = 1647831600  # 2022-03-21T11:00:00+08:00

# 1) Extract rows for the two timestamps
df_1059 = metric_df[metric_df['timestamp'] == ts_1059][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_1059'})
df_1100 = metric_df[metric_df['timestamp'] == ts_1100][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_1100'})

# 2) Keep only (cmdb_id, kpi_name) present at both timestamps (inner join)
merged = df_1059.merge(df_1100, on=['cmdb_id','kpi_name'], how='inner')

# 2b) Compute delta and percent_change (percent_change null if value_1059 == 0)
merged['delta'] = merged['value_1100'] - merged['value_1059']
merged['percent_change'] = np.where(
    merged['value_1059'] == 0,
    np.nan,
    (merged['delta'] / merged['value_1059']) * 100.0
)

# Add local time strings for the two epochs (same for all rows)
earliest_local = pd.to_datetime(ts_1059, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%dT%H:%M:%S%z')
latest_local = pd.to_datetime(ts_1100, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%dT%H:%M:%S%z')
merged['earliest_ts_local'] = earliest_local
merged['latest_ts_local'] = latest_local

# 3) Top 20 rows sorted by absolute delta (largest magnitude changes)
merged['abs_delta'] = merged['delta'].abs()
top20_changes = (
    merged
    .sort_values('abs_delta', ascending=False)
    .head(20)
    .loc[:, ['cmdb_id','kpi_name','value_1059','value_1100','delta','percent_change','earliest_ts_local','latest_ts_local']]
    .reset_index(drop=True)
)

# 4) Summary: how many distinct cmdb_id had any metric changes (delta != 0)
changed = merged[merged['delta'] != 0].copy()
num_distinct_cmdb_changed = int(changed['cmdb_id'].nunique())

# Top 5 cmdb_id by number of changed KPIs
top5_cmdb_by_changes = (
    changed
    .groupby('cmdb_id', as_index=False)
    .agg(number_of_changed_kpis=('kpi_name','nunique'))
    .sort_values('number_of_changed_kpis', ascending=False)
    .head(5)
    .reset_index(drop=True)
)

# Compact outputs
top20_changes, num_distinct_cmdb_changed, top5_cmdb_by_changes

```
Out[7]:
```
Summary of metric changes between 2022-03-21T10:59:00+08:00 and 2022-03-21T11:00:00+08:00:

What I compared
- Values at 1647831540 (10:59) vs 1647831600 (11:00) for (cmdb_id, kpi_name) pairs present at both timestamps.
- Returned the top 20 pairs ranked by absolute delta.

Top notable changes (top examples from the top-20):
- node-6:
  - system.disk.total increased by +3.2018e+09 (≈ +62.8%).
  - system.fs.inodes.total increased by +2.6941e+09 (≈ +70.7%).
  - system.fs.inodes.free increased by +2.6934e+09 (≈ +70.7%).
  - system.disk.used increased by +2.5967e+09 (≈ +70.7%).
  - system.disk.free increased by +6.0812e+08 (≈ +43.2%).
  => Large, simultaneous jumps in disk and inode metrics on node-6 — suggests a filesystem/state change or metric reporting anomaly on that node at 11:00.

- JVM / process metrics (adservice / adservice2):
  - adservice2 runtime.ProcessCpuTime +2.6583e+08 (+0.43%).
  - adservice runtime.ProcessCpuTime +1.9833e+08 (+0.009%).
  - adservice and adservice2 threading / cpu-time / alloc counters change by tens to hundreds of millions (small percent changes for large absolute counters).
  - adservice Eden memory used +1.4609e+06 (+11.54%); adservice2 Eden +1.0918e+06 (+20.90%).
  => JVM memory pools and thread/cpu counters moved notably for adservice/adservice2.

Other examples
- node-4, node-5 and node-1 show modest inode total/free decreases ~1.0–1.5e+06 (≈ -0.01% to -0.0067%).
- Several other runtime counters for adservice/adservice2 show smaller absolute changes.

Timestamps shown for comparisons
- earliest (10:59) local label: 2022-03-21T10:59:00+08:00
- latest  (11:00) local label: 2022-03-21T11:00:00+08:00

High-level counts and hotspots
- Distinct cmdb_id that had any metric change (delta != 0): 59
- Top 5 cmdb_id by number of changed KPIs:
  1. frontend-2 — 66 changed KPIs
  2. frontend-1 — 66 changed KPIs
  3. frontend-0 — 66 changed KPIs
  4. frontend2-0 — 60 changed KPIs
  5. checkoutservice-2 — 58 changed KPIs

Interpretation and recommended checks
- The largest, most suspicious deltas are on node-6 (big disk/inode jumps). Investigate node-6 for:
  - filesystem mounts/unmounts, storage reconfiguration, or metric-exporter/agent restart at ~11:00;
  - disk provisioning events or monitoring artifacts that could produce large jumps.
- JVM memory increases in adservice/adservice2 (Eden usage) and changes in thread/cpu counters warrant checking those services' logs and GC activity around 11:00.
- Large counts of changed KPIs on frontend pods (66 each) indicate a broad change affecting frontends at 11:00 — check deployment/agent restarts, config changes, or a surge/drop in traffic.
- Because many large changes coincide at the same timestamp, also check telemetry agent/collector and node-level events (restarts, reboots, metric exporter resets) at 2022-03-21T11:00:00+08:00.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name    value_1059    value_1100         delta  percent_change         earliest_ts_local           latest_ts_local
0       node-6                                  system.disk.total  5.101720e+09  8.303561e+09  3.201841e+09       62.760033  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
1       node-6                             system.fs.inodes.total  3.811664e+09  6.505770e+09  2.694106e+09       70.680576  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
2       node-6                              system.fs.inodes.free  3.810091e+09  6.503466e+09  2.693375e+09       70.690587  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
3       node-6                                   system.disk.used  3.671536e+09  6.268273e+09  2.596737e+09       70.726168  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
4       node-6                                   system.disk.free  1.409049e+09  2.017172e+09  6.081232e+08       43.158427  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
5   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  6.217750e+10  6.244333e+10  2.658333e+08        0.427539  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
6    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  2.192121e+12  2.192319e+12  1.983333e+08        0.009048  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
7    adservice  runtime.java_lang_Threading_CurrentThreadUserTime  2.102683e+11  2.100875e+11 -1.808333e+08       -0.086001  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
8    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  2.303237e+11  2.302513e+11 -7.245202e+07       -0.031457  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
9   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  4.691935e+09  4.741670e+09  4.973476e+07        1.060005  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
10  adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  4.346667e+09  4.392500e+09  4.583333e+07        1.054448  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
11  adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  3.078181e+09  3.093733e+09  1.555169e+07        0.505223  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
12   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  1.666013e+11  1.666153e+11  1.406483e+07        0.008442  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
13      node-4                             system.fs.inodes.total  1.135705e+10  1.135553e+10 -1.525760e+06       -0.013434  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
14      node-4                              system.fs.inodes.free  1.135327e+10  1.135174e+10 -1.525760e+06       -0.013439  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
15   adservice  runtime.java_lang_MemoryPool_Usage_used.Eden_S...  1.266225e+07  1.412318e+07  1.460935e+06       11.537721  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
16  adservice2  runtime.java_lang_MemoryPool_Usage_used.Eden_S...  5.224745e+06  6.316551e+06  1.091806e+06       20.896830  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
17      node-5                              system.fs.inodes.free  1.512490e+10  1.512388e+10 -1.021269e+06       -0.006752  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
18      node-5                             system.fs.inodes.total  1.512953e+10  1.512851e+10 -1.021269e+06       -0.006750  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800
19      node-1                             system.fs.inodes.total  1.135604e+10  1.135520e+10 -8.448000e+05       -0.007439  2022-03-21T10:59:00+0800  2022-03-21T11:00:00+0800, 59,              cmdb_id  number_of_changed_kpis
0         frontend-2                      66
1         frontend-1                      66
2         frontend-0                      66
3        frontend2-0                      60
4  checkoutservice-2                      58)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse metric_df, trace_df, log_df from the kernel
tz = pytz.timezone('Asia/Shanghai')
ts = 1647831600  # target epoch

# Ensure expected columns exist
assert {'timestamp','cmdb_id'}.issubset(metric_df.columns)
assert {'timestamp','cmdb_id'}.issubset(trace_df.columns)
assert {'timestamp','cmdb_id'}.issubset(log_df.columns)

def top_counts(df, key_col, name_col, limit=20):
    # key_col may contain NaN; cast to string for stable grouping
    keys = df[key_col].fillna('').astype(str)
    names = df[name_col].fillna('').astype(str)
    top_keys = keys.value_counts().head(limit)
    top_names = names.value_counts().head(limit)
    top_keys_list = [{"cmdb_id": k, "count": int(v)} for k,v in top_keys.items()]
    top_names_list = [{"name": n, "count": int(v)} for n,v in top_names.items()]
    return top_keys_list, top_names_list

# Filter each dataframe to the timestamp
metric_at = metric_df[metric_df['timestamp'] == ts].copy()
trace_at = trace_df[trace_df['timestamp'] == ts].copy()
log_at = log_df[log_df['timestamp'] == ts].copy()

# metric.csv summary
metric_total = int(metric_at.shape[0])
metric_top_cmdb, metric_top_kpi = top_counts(metric_at, 'cmdb_id', 'kpi_name', limit=20)

metric_summary = {
    "total_rows_at_timestamp": metric_total,
    "top_20_cmdb_id_with_counts": metric_top_cmdb,
    "top_20_name_with_counts": metric_top_kpi
}

# trace.csv summary
trace_total = int(trace_at.shape[0])
trace_top_cmdb, trace_top_names = top_counts(trace_at, 'cmdb_id', 'trace_name', limit=20)

# extra trace counts
trace_row_count_count = int(trace_at['trace_name'].str.contains('row_count', na=False).sum())
trace_from_root_count = int(trace_at['trace_name'].str.contains('from_root', na=False).sum())

trace_summary = {
    "total_rows_at_timestamp": trace_total,
    "top_20_cmdb_id_with_counts": trace_top_cmdb,
    "top_20_name_with_counts": trace_top_names,
    "count_trace_name_contains_row_count": trace_row_count_count,
    "count_trace_name_contains_from_root": trace_from_root_count
}

# log.csv summary
log_total = int(log_at.shape[0])
log_top_cmdb, log_top_names = top_counts(log_at, 'cmdb_id', 'log_name', limit=20)

log_summary = {
    "total_rows_at_timestamp": log_total,
    "top_20_cmdb_id_with_counts": log_top_cmdb,
    "top_20_name_with_counts": log_top_names
}

result = {
    "metric": metric_summary,
    "trace": trace_summary,
    "log": log_summary
}

result

```
Out[8]:
```
Summary for timestamp 1647831600 (2022-03-21T11:00:00+08:00):

- metric.csv
  - Total rows at this timestamp: 5,288.
  - Top components by metric count (examples, up to 20): adservice (341), adservice2 (333), frontend2-0 (200), frontend-2 (188), frontend-1 (188), frontend-0 (188), checkoutservice2-0 (148), checkoutservice-1 (144), checkoutservice-2 (140), checkoutservice-0 (140), productcatalogservice-1 (120), productcatalogservice-0 (120), productcatalogservice-2 (116), ...
  - Top KPI names (examples): many container/node-6 metrics dominate (e.g., container.node-6.container_network_transmit_errors.eth0, container.node-6.container_network_transmit_packets.eth0, container.node-6.container_network_receive_MB.eth0, etc.), each appearing multiple times (counts ~34–36). Node-6 container metrics are highly represented at this timestamp.

- trace.csv
  - Total rows at this timestamp: 0 (no trace records found).
  - Consequently no top cmdb_id or trace_name counts; counts for trace_name containing 'row_count' and 'from_root' are both 0.

- log.csv
  - Total rows at this timestamp: 30.
  - Top components by log count (examples): adservice-1 (2), cartservice-0/1/2 (2 each), currencyservice-0/1/2 (2 each), frontend-0 (2), frontend-1 (2), productcatalogservice-2 (2), recommendationservice-0/1/2 (2 each), shippingservice-0/1 (2 each), ...
  - Top log names: log.error_count (15 rows) and log.row_count (15 rows).

Takeaway
- Metrics are plentiful at 11:00 with strong representation from adservice/adservice2, frontends and node-6 container KPIs (node-6 metrics especially prominent).
- No trace data is present at this timestamp (trace.csv empty), which may indicate tracing data not captured/sent at 11:00.
- Logs are sparse (30 rows) but evenly split between error_count and row_count entries.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'total_rows_at_timestamp': 5288, 'top_20_cmdb_id_with_counts': [{'cmdb_id': 'adservice', 'count': 341}, {'cmdb_id': 'adservice2', 'count': 333}, {'cmdb_id': 'frontend2-0', 'count': 200}, {'cmdb_id': 'frontend-2', 'count': 188}, {'cmdb_id': 'frontend-1', 'count': 188}, {'cmdb_id': 'frontend-0', 'count': 188}, {'cmdb_id': 'checkoutservice2-0', 'count': 148}, {'cmdb_id': 'checkoutservice-1', 'count': 144}, {'cmdb_id': 'checkoutservice-2', 'count': 140}, {'cmdb_id': 'checkoutservice-0', 'count': 140}, {'cmdb_id': 'productcatalogservice-1', 'count': 120}, {'cmdb_id': 'productcatalogservice-0', 'count': 120}, {'cmdb_id': 'productcatalogservice-2', 'count': 116}, {'cmdb_id': 'productcatalogservice2-0', 'count': 104}, {'cmdb_id': 'adservice2-0', 'count': 102}, {'cmdb_id': 'recommendationservice2-0', 'count': 102}, {'cmdb_id': 'cartservice2-0', 'count': 100}, {'cmdb_id': 'cartservice-2', 'count': 100}, {'cmdb_id': 'cartservice-0', 'count': 100}, {'cmdb_id': 'recommendationservice-2', 'count': 98}], 'top_20_name_with_counts': [{'name': 'container.node-6.container_network_transmit_errors.eth0', 'count': 36}, {'name': 'container.node-6.container_network_transmit_packets.eth0', 'count': 36}, {'name': 'container.node-6.container_network_transmit_packets_dropped.eth0', 'count': 36}, {'name': 'container.node-6.container_network_receive_MB.eth0', 'count': 36}, {'name': 'container.node-6.container_network_receive_errors.eth0', 'count': 36}, {'name': 'container.node-6.container_network_receive_packets.eth0', 'count': 36}, {'name': 'container.node-6.container_network_receive_packets_dropped.eth0', 'count': 36}, {'name': 'container.node-6.container_network_transmit_MB.eth0', 'count': 36}, {'name': 'container.node-6.container_spec_cpu_shares', 'count': 34}, {'name': 'container.node-6.container_spec_memory_reservation_limit_MB', 'count': 34}, {'name': 'container.node-6.container_spec_cpu_quota', 'count': 34}, {'name': 'container.node-6.container_spec_memory_swap_limit_MB', 'count': 34}, {'name': 'container.node-6.container_fs_writes./dev/vda1', 'count': 34}, {'name': 'container.node-6.container_fs_writes./dev/vda', 'count': 34}, {'name': 'container.node-6.container_memory_usage_MB', 'count': 34}, {'name': 'container.node-6.container_memory_swap', 'count': 34}, {'name': 'container.node-6.container_fs_limit_MB./dev/vda1', 'count': 34}, {'name': 'container.node-6.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 34}, {'name': 'container.node-6.container_fs_io_time_seconds./dev/vda1', 'count': 34}, {'name': 'container.node-6.container_fs_io_current./dev/vda1', 'count': 34}]}, 'trace': {'total_rows_at_timestamp': 0, 'top_20_cmdb_id_with_counts': [], 'top_20_name_with_counts': [], 'count_trace_name_contains_row_count': 0, 'count_trace_name_contains_from_root': 0}, 'log': {'total_rows_at_timestamp': 30, 'top_20_cmdb_id_with_counts': [{'cmdb_id': 'adservice-1', 'count': 2}, {'cmdb_id': 'cartservice-0', 'count': 2}, {'cmdb_id': 'cartservice-1', 'count': 2}, {'cmdb_id': 'cartservice-2', 'count': 2}, {'cmdb_id': 'currencyservice-0', 'count': 2}, {'cmdb_id': 'currencyservice-1', 'count': 2}, {'cmdb_id': 'currencyservice-2', 'count': 2}, {'cmdb_id': 'frontend-0', 'count': 2}, {'cmdb_id': 'frontend-1', 'count': 2}, {'cmdb_id': 'productcatalogservice-2', 'count': 2}, {'cmdb_id': 'recommendationservice-0', 'count': 2}, {'cmdb_id': 'recommendationservice-1', 'count': 2}, {'cmdb_id': 'recommendationservice-2', 'count': 2}, {'cmdb_id': 'shippingservice-0', 'count': 2}, {'cmdb_id': 'shippingservice-1', 'count': 2}], 'top_20_name_with_counts': [{'name': 'log.error_count', 'count': 15}, {'name': 'log.row_count', 'count': 15}]}}```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Timestamps
ts_1059 = 1647831540
ts_1100 = 1647831600

# 1) Extract rows for the two timestamps
df_two_times = metric_df[metric_df['timestamp'].isin([ts_1059, ts_1100])].copy()

# 2) Restrict to kpi_name containing any of the specified substrings (case-insensitive)
substrs = [
    'container_start_time',
    'container.start_time',
    'system.disk',
    'system.fs.inodes',
    'container_fs_usage',
    'container_fs_reads',
    'container_fs_writes',
    'container_memory_usage_MB'
]
# build regex safely (escape dots)
regex = '(' + '|'.join(re.escape(s) for s in substrs) + ')'
mask_kpis = df_two_times['kpi_name'].str.contains(regex, case=False, na=False)
df_sel = df_two_times[mask_kpis].copy()

# Separate rows at each timestamp
df_1059 = df_sel[df_sel['timestamp'] == ts_1059][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_1059'})
df_1100 = df_sel[df_sel['timestamp'] == ts_1100][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_1100'})

# 3) Full outer join on (cmdb_id, kpi_name) to include pairs present at either timestamp
merged = pd.merge(df_1059, df_1100, on=['cmdb_id','kpi_name'], how='outer')

# Ensure numeric dtype
merged['value_1059'] = pd.to_numeric(merged['value_1059'], errors='coerce')
merged['value_1100'] = pd.to_numeric(merged['value_1100'], errors='coerce')

# 4) Compute delta and percent_change (percent_change null if value_1059 == 0 or NaN)
merged['delta'] = merged['value_1100'] - merged['value_1059']
merged['percent_change'] = np.where(
    (merged['value_1059'].isna()) | (merged['value_1059'] == 0),
    np.nan,
    (merged['delta'] / merged['value_1059']) * 100.0
)

# 5) Top 50 rows sorted by absolute delta (largest magnitude)
merged['abs_delta'] = merged['delta'].abs().fillna(0)
top50_changes = (
    merged
    .sort_values('abs_delta', ascending=False)
    .head(50)
    .loc[:, ['cmdb_id','kpi_name','value_1059','value_1100','delta','percent_change']]
    .reset_index(drop=True)
)

# 6) Aggregate: container_start_time related changes
start_mask = merged['kpi_name'].str.contains('container_start_time|container.start_time', case=False, na=False)
start_changes = merged[start_mask].copy()

# Define "change" as a non-null delta that is not zero
start_changes['changed'] = start_changes['delta'].notna() & (start_changes['delta'] != 0)

changed_start = start_changes[start_changes['changed']]

# Count distinct cmdb_id and list them (limit top 20)
start_change_count = int(changed_start['cmdb_id'].nunique())
start_change_cmdbs = (
    changed_start
    .groupby('cmdb_id', as_index=False)
    .agg(number_of_changed_kpis=('kpi_name','nunique'))
    .sort_values('number_of_changed_kpis', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Final compact outputs (DataFrames / scalars)
top50_changes, start_change_count, start_change_cmdbs

```
Out[9]:
```
Summary of selected storage/container metrics between 2022-03-21T10:59 and 2022-03-21T11:00 (UTC+8):

What was compared
- Values at 10:59 vs 11:00 for KPIs matching: container_start_time / container.start_time, system.disk*, system.fs.inodes*, container_fs_usage*, container_fs_reads*, container_fs_writes*, container_memory_usage_MB.

Top changes (largest absolute deltas)
- node-6 shows the largest, most significant jumps at 11:00:
  - system.disk.total increased by +3.20e+09 (+62.8%).
  - system.fs.inodes.total increased by +2.69e+09 (+70.7%).
  - system.fs.inodes.free increased by +2.69e+09 (+70.7%).
  - system.disk.used increased by +2.60e+09 (+70.7%).
  - system.disk.free increased by +6.08e+08 (+43.2%).
- Several JVM/container FS usage and memory usage KPIs across many services have small absolute changes (fractions to a few units), for example container_fs_usage_MB deltas on various cmdb_id are small (0.05–0.1 MB).
- A handful of JVM/container memory usage metrics for adservice/adservice2 and others show modest increases (e.g., Eden used increases noted previously).

Container start times
- No cmdb_id showed a change in any 'container_start_time' / 'container.start_time' KPI between the two timestamps (count = 0).

High-level interpretation
- The dominant and suspicious signals are the very large, concurrent jumps in disk and inode metrics on node-6 at 11:00. Such simultaneous big jumps usually indicate a node-level event or metric reporting artifact (e.g., filesystem remount, storage reconfiguration, metric-exporter restart, or a monitoring/aggregation glitch) rather than normal per-process behavior.
- Other KPI deltas are small and likely normal noise except where noted above.

Recommended immediate checks
1. Inspect node-6 system events, storage system, and exporter/agent logs around 11:00 for mounts, reconfiguration, disk operations, or agent restarts.
2. Confirm whether disk capacity or inode counts were actually changed by the storage backend (or if this is a telemetry/reporting artifact).
3. If node-6 changes are confirmed real, prioritize investigating I/O, mounts, and any jobs that could alter inode/disk counts at that time.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                          kpi_name    value_1059    value_1100         delta  percent_change
0               node-6                                 system.disk.total  5.101720e+09  8.303561e+09  3.201841e+09       62.760033
1               node-6                            system.fs.inodes.total  3.811664e+09  6.505770e+09  2.694106e+09       70.680576
2               node-6                             system.fs.inodes.free  3.810091e+09  6.503466e+09  2.693375e+09       70.690587
3               node-6                                  system.disk.used  3.671536e+09  6.268273e+09  2.596737e+09       70.726168
4               node-6                                  system.disk.free  1.409049e+09  2.017172e+09  6.081232e+08       43.158427
..                 ...                                               ...           ...           ...           ...             ...
45   currencyservice-1  container.node-6.container_fs_usage_MB./dev/vda1  7.716016e+01  7.725781e+01  9.765625e-02        0.126563
46       cartservice-0        container.node-6.container_memory_usage_MB  2.113477e+02  2.114277e+02  8.007812e-02        0.037889
47         frontend2-0  container.node-6.container_fs_usage_MB./dev/vda1  6.569531e+01  6.576562e+01  7.031250e-02        0.107028
48  currencyservice2-0  container.node-6.container_fs_usage_MB./dev/vda1  6.750508e+02  6.751191e+02  6.835938e-02        0.010127
49          frontend-0  container.node-6.container_fs_usage_MB./dev/vda1  2.756836e+01  2.762500e+01  5.664062e-02        0.205455

[50 rows x 6 columns], 0, Empty DataFrame
Columns: [cmdb_id, number_of_changed_kpis]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```